# 🧬 SHAP Analysis for Clinical Model Interpretability
## Thyroid Cancer Recurrence Prediction

---

This notebook performs **SHAP (SHapley Additive exPlanations)** analysis on a heterogeneous stacking ensemble model trained to predict thyroid cancer recurrence.

### Objectives
- Build and train a stacking ensemble classifier (LR + SVM + RF) on the Thyroid Differentiated dataset
- Apply KernelSHAP to explain the black-box ensemble
- Derive **global feature importance** rankings
- Generate **individual patient-level explanations**
- Produce clinically interpretable outputs for decision support

### Dataset
| Property | Value |
|---|---|
| File | `Thyroid_Diff.csv` |
| Target | `Recurred` (Yes / No → 1 / 0) |
| Task | Binary Classification |

> **Note:** All outputs (plots, CSVs) are saved to `../results/SHAP_Analysis_Outputs/`

---
## 1. Imports & Environment Setup

Load all required libraries and configure output directories.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
output_folder = os.path.join(project_root, "results", "SHAP_Analysis_Outputs")
os.makedirs(output_folder, exist_ok=True)

print(f"Current directory: {current_dir}")
print(f"Output folder: {output_folder}")
print("✅ SHAP ANALYSIS FOR CLINICAL MODEL INTERPRETABILITY")

---
## 2. Load Dataset

Read the thyroid differentiated cancer dataset and separate features from the target variable (`Recurred`).

- **Class 1** → Recurrence  
- **Class 0** → No Recurrence

In [ ]:
data = pd.read_csv("Thyroid_Diff.csv")
X = data.drop("Recurred", axis=1)
y = data["Recurred"].map({"No": 0, "Yes": 1})

print("✅ DATA LOADED")
print(f"Total samples      : {len(data)}")
print(f"Recurrence (1)     : {sum(y==1)}")
print(f"Non-recurrence (0) : {sum(y==0)}")
print(f"Features           : {list(X.columns)}")

---
## 3. Encode Categorical Variables

Apply **one-hot encoding** (`pd.get_dummies`) to all categorical features.  
`drop_first=False` retains all dummy columns for interpretability — important for SHAP analysis.

In [ ]:
X_encoded = pd.get_dummies(X, drop_first=False)

print(f"Original features : {X.shape[1]}")
print(f"Encoded features  : {X_encoded.shape[1]}")

---
## 4. Train / Test Split

Split the dataset into **75% training** and **25% test** sets.

- `stratify=y` ensures class balance is preserved in both splits  
- `random_state=321` for reproducibility

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    train_size=0.75,
    random_state=321,
    stratify=y
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

---
## 5. Feature Scaling

Apply **StandardScaler** (zero mean, unit variance) to normalise features.

> ⚠️ The scaler is **fit only on training data** and then applied to the test set to prevent data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Data scaled using StandardScaler")
print(f"Training shape : {X_train_scaled.shape}")
print(f"Test shape     : {X_test_scaled.shape}")

---
## 6. Build & Train Heterogeneous Stacking Ensemble

The ensemble consists of **three diverse base learners**, each paired with a distinct dimensionality reduction step:

| Pipeline | Reducer | Classifier |
|---|---|---|
| `lr_pca` | PCA (5 components) | Logistic Regression (L1) |
| `svm_tsvd` | TruncatedSVD (5 components) | SVM (sigmoid kernel) |
| `rf` | UMAP / PCA fallback | Random Forest (entropy) |

A **Logistic Regression meta-learner** combines base predictions via 3-fold cross-validation.

> 💡 UMAP is used for non-linear reduction if available; PCA is the fallback.

In [ ]:
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.linear_model import LogisticRegression as LR
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier as RF
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import Pipeline

print("🔧 BUILDING STACKING ENSEMBLE")

try:
    from umap import UMAP
    umap_available = True
    print("✅ UMAP available — using non-linear dimensionality reduction")
except ImportError:
    umap_available = False
    print("⚠️  UMAP not available — using PCA as fallback")

# --- Pipeline 1: PCA + Logistic Regression ---
pca = PCA(n_components=5)
lr_pca = LR(C=0.36, penalty='l1', solver='liblinear',
            class_weight='balanced', random_state=42, max_iter=1000)
pipeline_lr_pca = Pipeline([('pca', pca), ('clf', lr_pca)])

# --- Pipeline 2: TruncatedSVD + SVM ---
tsvd = TruncatedSVD(n_components=5)
svm_tsvd = SVC(probability=True, C=0.25, kernel='sigmoid',
               class_weight='balanced', random_state=42)
pipeline_svm_tsvd = Pipeline([('tsvd', tsvd), ('clf', svm_tsvd)])

# --- Pipeline 3: UMAP/PCA + Random Forest ---
if umap_available:
    umap_reducer = UMAP(n_components=5, random_state=42, n_neighbors=15, min_dist=0.1)
    rf_model = RF(criterion='entropy', max_depth=None, max_features='log2',
                  min_samples_leaf=4, n_estimators=200,
                  class_weight='balanced', random_state=42)
    pipeline_rf = Pipeline([('umap', umap_reducer), ('clf', rf_model)])
else:
    pca_rf = PCA(n_components=5)
    rf_model = RF(criterion='entropy', max_depth=None, max_features='log2',
                  min_samples_leaf=4, n_estimators=200,
                  class_weight='balanced', random_state=42)
    pipeline_rf = Pipeline([('pca', pca_rf), ('clf', rf_model)])

# --- Stacking Ensemble ---
ensemble_model = StackingClassifier(
    estimators=[
        ('lr_pca',   pipeline_lr_pca),
        ('svm_tsvd', pipeline_svm_tsvd),
        ('rf',       pipeline_rf)
    ],
    final_estimator=LR(C=0.1, class_weight='balanced', random_state=42),
    cv=3,
    stack_method='predict_proba',
    n_jobs=1
)

print("⏳ Training stacking ensemble...")
ensemble_model.fit(X_train_scaled, y_train)

from sklearn.metrics import balanced_accuracy_score
y_pred = ensemble_model.predict(X_test_scaled)
test_bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f"\n✅ Ensemble Model Test Balanced Accuracy: {test_bal_acc:.4f}")

---
## 7. SHAP Analysis — KernelExplainer

**KernelSHAP** is used because the stacking ensemble is a black-box model with no tree structure.  
It approximates Shapley values by fitting a weighted linear model around the prediction function.

| Parameter | Value | Rationale |
|---|---|---|
| Background samples | 30 | Representative summary of training distribution |
| Explanation samples | 50 | Test instances to explain |
| `nsamples` | 50 | SHAP perturbation samples per instance |

> ⏱️ **This cell may take 2–3 minutes** to complete due to KernelSHAP's perturbation-based computation.

In [ ]:
import shap

print("🔍 SHAP ANALYSIS: HETEROGENEOUS STACKING ENSEMBLE\n")

feature_names = X_encoded.columns.tolist()

def predict_proba_wrapper(X_scaled):
    return ensemble_model.predict_proba(X_scaled)

background_size = min(30, len(X_train_scaled))
explain_size    = min(50, len(X_test_scaled))

background_sample = X_train_scaled[
    np.random.choice(len(X_train_scaled), background_size, replace=False)
]
explain_sample = X_test_scaled[
    np.random.choice(len(X_test_scaled), explain_size, replace=False)
]

print(f"Background samples  : {background_sample.shape[0]}")
print(f"Explanation samples : {explain_sample.shape[0]}")
print("⏳ Creating SHAP explainer (may take 2–3 minutes)...")

explainer = shap.KernelExplainer(predict_proba_wrapper, background_sample)

print("⏳ Calculating SHAP values...")
shap_values = explainer.shap_values(explain_sample, nsamples=50)

print("✅ SHAP analysis complete!")

# Extract SHAP values for class 1 (Recurrence)
if len(shap_values.shape) == 3:
    shap_values_class1 = shap_values[:, :, 1]
else:
    shap_values_class1 = shap_values

---
## 8. Global Feature Importance

Compute **mean absolute SHAP values** across all explanation samples to rank features by their average contribution to model predictions.

Two visualisations are generated:
1. **Bar chart** — magnitude of influence (top 15 features)
2. **Beeswarm plot** — direction and distribution of impact per feature

In [ ]:
print("📊 GLOBAL FEATURE IMPORTANCE\n")

mean_abs_shap = np.abs(shap_values_class1).mean(axis=0)
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Mean_SHAP_Value': mean_abs_shap
}).sort_values('Mean_SHAP_Value', ascending=False)

print("Top 15 Most Influential Features:")
print("-" * 55)
for i, row in feature_importance.head(15).iterrows():
    feat = row['Feature'][:40] if len(row['Feature']) > 40 else row['Feature']
    print(f"  {feat:<40}: {row['Mean_SHAP_Value']:.4f}")

# --- Bar Chart ---
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(top_features['Feature'], top_features['Mean_SHAP_Value'], color='steelblue')
plt.xlabel('Mean |SHAP Value|', fontsize=11)
plt.ylabel('Feature', fontsize=11)
plt.title('Top 15 Most Influential Features — Ensemble Model', fontsize=13)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "shap_ensemble_global_importance.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("[SAVED] shap_ensemble_global_importance.png")

# --- Beeswarm Plot ---
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values_class1, explain_sample,
                  feature_names=feature_names, plot_type="dot",
                  max_display=15, show=False)
plt.title("SHAP Beeswarm Plot — Feature Impact Direction", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "shap_ensemble_beeswarm.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("[SAVED] shap_ensemble_beeswarm.png")

---
## 9. Save Feature Importance to CSV

Export the ranked feature importance table for downstream analysis or reporting.

In [ ]:
feature_importance.to_csv(
    os.path.join(output_folder, "shap_feature_importance_ensemble.csv"),
    index=False
)
print("[SAVED] shap_feature_importance_ensemble.csv")
feature_importance.head(10)

---
## 10. Individual Patient Explanations

SHAP enables **local (instance-level) explanations** — showing which features pushed the model toward or away from a recurrence prediction for a specific patient.

Two patients are selected from the test set:
- 🔴 **Recurrence patient** — actual label = 1
- 🟢 **Non-recurrence patient** — actual label = 0

For each patient, the top features driving the prediction are identified.

In [ ]:
print("👤 INDIVIDUAL PATIENT EXPLANATIONS\n")

y_test_array = y_test.values if hasattr(y_test, 'values') else y_test
recurrence_indices     = np.where(y_test_array == 1)[0]
non_recurrence_indices = np.where(y_test_array == 0)[0]

if len(recurrence_indices) > 0 and len(non_recurrence_indices) > 0:
    recurrence_idx     = recurrence_indices[0]
    non_recurrence_idx = non_recurrence_indices[0]

    recurrence_patient     = X_test_scaled[recurrence_idx:recurrence_idx+1]
    non_recurrence_patient = X_test_scaled[non_recurrence_idx:non_recurrence_idx+1]

    pred_recurrence     = ensemble_model.predict_proba(recurrence_patient)[0, 1]
    pred_non_recurrence = ensemble_model.predict_proba(non_recurrence_patient)[0, 1]

    print(f"🔴 RECURRENCE PATIENT (Actual: Recurrence)")
    print(f"   Predicted recurrence probability : {pred_recurrence:.4f}")
    print(f"\n🟢 NON-RECURRENCE PATIENT (Actual: No Recurrence)")
    print(f"   Predicted recurrence probability : {pred_non_recurrence:.4f}")

    shap_recurrence     = explainer.shap_values(recurrence_patient, nsamples=50)
    shap_non_recurrence = explainer.shap_values(non_recurrence_patient, nsamples=50)

    if len(shap_recurrence.shape) == 3:
        shap_rec_class1 = shap_recurrence[0, :, 1].flatten()
        shap_non_class1 = shap_non_recurrence[0, :, 1].flatten()
    elif len(shap_recurrence.shape) == 2:
        shap_rec_class1 = shap_recurrence[0].flatten()
        shap_non_class1 = shap_non_recurrence[0].flatten()
    else:
        shap_rec_class1 = shap_recurrence.flatten()
        shap_non_class1 = shap_non_recurrence.flatten()

    if isinstance(explainer.expected_value, list):
        expected_value = explainer.expected_value[1]
    elif hasattr(explainer.expected_value, '__len__') and len(explainer.expected_value) > 1:
        expected_value = explainer.expected_value[1]
    else:
        expected_value = explainer.expected_value

    print(f"\n  Debug Info:")
    print(f"    SHAP values shape  : {shap_rec_class1.shape}")
    print(f"    Features count     : {len(feature_names)}")
    print(f"    Expected value type: {type(expected_value)}")

    # Save patient SHAP values
    shap_patient_df = pd.DataFrame({
        'Feature': feature_names,
        'SHAP_Recurrence_Patient':    shap_rec_class1,
        'SHAP_NonRecurrence_Patient': shap_non_class1
    })
    shap_patient_df.to_csv(os.path.join(output_folder, "patient_shap_values.csv"), index=False)
    print("\n[SAVED] patient_shap_values.csv")

    # Top drivers
    top_positive_idx = np.argsort(shap_rec_class1)[-5:][::-1]
    top_negative_idx = np.argsort(shap_rec_class1)[:5]

    print("\n  🔺 TOP 5 FEATURES PUSHING TOWARD RECURRENCE:")
    for idx in top_positive_idx:
        if shap_rec_class1[idx] > 0:
            print(f"    {feature_names[idx][:45]}: +{shap_rec_class1[idx]:.4f}")

    print("\n  🔻 TOP 5 FEATURES PUSHING AWAY FROM RECURRENCE:")
    for idx in top_negative_idx:
        if shap_rec_class1[idx] < 0:
            print(f"    {feature_names[idx][:45]}: {shap_rec_class1[idx]:.4f}")

    # Patient feature bar chart
    try:
        plt.figure(figsize=(10, 6))
        top_features_idx = top_positive_idx[:5]
        top_values = shap_rec_class1[top_features_idx]
        top_names  = [feature_names[i][:35] for i in top_features_idx]
        colors = ['red' if v > 0 else 'green' for v in top_values]
        plt.barh(top_names, top_values, color=colors)
        plt.xlabel('SHAP Value (Impact on Recurrence Prediction)')
        plt.title(f'Top Features — Recurrence Patient (P(Recurrence) = {pred_recurrence:.3f})')
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, "shap_recurrence_patient_top_features.png"),
                    dpi=150, bbox_inches='tight')
        plt.show()
        print("[SAVED] shap_recurrence_patient_top_features.png")
    except Exception as e:
        print(f"  ⚠️  Could not generate plot: {e}")

    # Save patient summary
    patient_summary = pd.DataFrame({
        'Patient_Type':          ['Recurrence', 'Non-Recurrence'],
        'Prediction_Probability': [pred_recurrence, pred_non_recurrence],
        'Actual_Class':          [1, 0]
    })
    patient_summary.to_csv(os.path.join(output_folder, "patient_predictions.csv"), index=False)
    print("[SAVED] patient_predictions.csv")
else:
    print("⚠️  Insufficient samples for patient-level explanations")

---
## 11. Clinical Interpretation Summary

Synthesise findings from the SHAP analysis into actionable clinical insights.

In [ ]:
print("📋 CLINICAL INTERPRETATION SUMMARY")
print("=" * 50)
print("""
KEY CLINICAL INSIGHTS FROM SHAP ANALYSIS:
-----------------------------------------

1. TOP PREDICTORS ALIGN WITH CLINICAL KNOWLEDGE:
   - Response to treatment (Structural Incomplete, Excellent)
   - Risk category (Low, High)
   - N stage (N1b)
   - Adenopathy (No, Bilateral)

2. MODEL BEHAVIOUR IS CLINICALLY SENSIBLE:
   - Structural Incomplete response → pushes toward recurrence
   - Excellent response            → pushes away from recurrence
   - High risk + N1b stage         → positively correlate with recurrence

3. SHAP TRANSPARENCY ENABLES CLINICIANS TO:
   - Verify model decisions align with medical guidelines
   - Identify which features drive each patient's prediction
   - Build trust in AI-assisted clinical decision making
""")

print("=" * 50)
print("✅ SHAP ANALYSIS COMPLETE")
print(f"📁 All outputs saved to: {output_folder}")

---

## 📁 Output Files Summary

| File | Description |
|---|---|
| `shap_ensemble_global_importance.png` | Bar chart of top 15 features by mean SHAP |
| `shap_ensemble_beeswarm.png` | Beeswarm plot showing impact direction |
| `shap_feature_importance_ensemble.csv` | Full ranked feature importance table |
| `patient_shap_values.csv` | Per-patient SHAP values for both example cases |
| `shap_recurrence_patient_top_features.png` | Top drivers for the recurrence patient |
| `patient_predictions.csv` | Prediction probabilities for both example patients |

---

> **Methodology Reference:** Lundberg, S. M., & Lee, S. I. (2017). *A unified approach to interpreting model predictions.* NeurIPS.